## Ważne linki (mogą przybliżyć dataset i trochę ułatwić nam zadanie):
https://www.sporthorizon.co.uk/blog/working-with-statsbomb-data-in-python
https://medium.com/@lucascarrasquillaparra/complete-guide-on-working-with-the-statsbomb-open-data-dataset-a57c26d5852b

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install statsbombpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.4 MB/s eta 0:00:00


In [4]:
from statsbombpy import sb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## "Główny" dataframe

In [5]:
comp_df = sb.competitions()

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [6]:
comp_df.head(20)

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,None,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,None,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,None,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,None,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,None,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,None,2024-02-12T12:47:34.340413


In [7]:
comp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   competition_id             80 non-null     int64 
 1   season_id                  80 non-null     int64 
 2   country_name               80 non-null     object
 3   competition_name           80 non-null     object
 4   competition_gender         80 non-null     object
 5   competition_youth          80 non-null     bool  
 6   competition_international  80 non-null     bool  
 7   season_name                80 non-null     object
 8   match_updated              80 non-null     object
 9   match_updated_360          58 non-null     object
 10  match_available_360        12 non-null     object
 11  match_available            80 non-null     object
dtypes: bool(2), int64(2), object(8)
memory usage: 6.5+ KB


To jest "główny" df naszego datasetu, raczej w kontekscie tego co chcemy zrobić, to nie jest on jakoś bardzo istotny, przez to, że obchodzi nas głównie xG, czyli raczej eventy z konkretnego meczu - więc w sumie jedyne "ważne" dla nas kolumny to: competition_id,, season_id (i może country_name i competition_name, ale to też raczej MOŻE z dużych liter).


In [29]:
len(pd.unique(comp_df['competition_name']))

24

W darmowej wersji datasetu mamy dostęp do danych z 24 różnych rozgrywek, z czego np.

In [53]:
print("-" ,len(pd.unique(comp_df['season_name'][comp_df['competition_name'] == 'Champions League'])), "sezonów w Lidze Mistrzów")
print("-" ,len(pd.unique(comp_df['season_name'][comp_df['competition_name'] == '1. Bundesliga'])), "sezony w Bundeslidze")
print("-" ,len(pd.unique(comp_df['season_name'][comp_df['competition_name'] == 'La Liga'])), "sezonów w La Lidze")


- 18 sezonów w Lidze Mistrzów
- 2 sezony w Bundeslidze
- 18 sezonów w La Lidze


In [68]:
comp_df[comp_df['competition_name'] == 'La Liga'].head(5) # przykładowo, choć dalej będzie przykład bundesligowy

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
40,11,90,Spain,La Liga,male,False,False,2020/2021,2025-11-02T03:55:19.577958,2025-11-02T03:55:53.923193,2025-11-02T03:55:53.923193,2025-11-02T03:55:19.577958
41,11,42,Spain,La Liga,male,False,False,2019/2020,2024-12-16T16:51:06.833054,2021-06-13T16:17:31.694,None,2024-12-16T16:51:06.833054
42,11,4,Spain,La Liga,male,False,False,2018/2019,2026-04-06T04:49:09.959011,2021-07-09T14:53:22.103024,None,2026-04-06T04:49:09.959011
43,11,1,Spain,La Liga,male,False,False,2017/2018,2025-07-14T10:01:16.674864,2021-06-13T16:17:31.694,None,2025-07-14T10:01:16.674864
44,11,2,Spain,La Liga,male,False,False,2016/2017,2025-03-09T16:03:54.962718,2021-06-13T16:17:31.694,None,2025-03-09T16:03:54.962718


In [76]:
comp_df['competition_name'].mode()

,competition_name
0,Champions League
1,La Liga


Najliczniej w datasecie występują dane z Champions League i La Liga.

## Coraz bliżej naszego "mięsa"

In [10]:
matches_presentation_df = sb.matches(competition_id=9, season_id=27) # wybór meczów z danych rozgrywek i danego sezonu

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [9]:
matches_presentation_df.head(20)

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,3890563,2016-05-14,15:30:00.000,3,2,available,unscheduled,2023-08-05T09:38:01.494282,None,34,...,Germany,44,Ralph Hasenhüttl,None,1967-08-09,15,Austria,1.1.0,2,2
1,3890550,2016-05-07,15:30:00.000,2,1,available,unscheduled,2023-10-28T23:30:33.200742,None,33,...,Germany,3980,Roger Schmidt,None,1967-03-13,85,Germany,1.1.0,2,2
2,3890540,2016-04-30,18:30:00.000,2,1,available,unscheduled,2023-07-24T07:41:44.627113,None,32,...,Germany,51,Pál Dárdai,None,1976-03-16,103,Hungary,1.1.0,2,2
3,3890534,2016-04-23,18:30:00.000,2,3,available,unscheduled,2023-07-19T17:07:48.502966,None,31,...,Germany,3980,Roger Schmidt,None,1967-03-13,85,Germany,1.1.0,2,2
4,3890521,2016-04-16,15:30:00.000,3,0,available,unscheduled,2023-07-25T09:12:32.507875,None,30,...,Germany,374,Niko Kovač,None,1971-10-15,56,Croatia,1.1.0,2,2
5,3890519,2016-04-10,17:30:00.000,0,2,available,unscheduled,2023-07-31T12:19:56.474040,None,29,...,Austria,3980,Roger Schmidt,None,1967-03-13,85,Germany,1.1.0,2,2
6,3890502,2016-04-01,20:30:00.000,3,0,available,unscheduled,2023-08-03T13:07:07.427967,None,28,...,Germany,411,Dieter Hecking,None,1964-09-12,85,Germany,1.1.0,2,2
7,3890500,2016-03-20,16:30:00.000,0,2,available,unscheduled,2023-07-19T13:10:23.962185,None,27,...,Germany,3980,Roger Schmidt,None,1967-03-13,85,Germany,1.1.0,2,2
8,3890492,2016-03-13,16:30:00.000,1,0,available,unscheduled,2023-08-03T13:06:19.003897,None,26,...,Germany,446,Bruno Labbadia,None,1966-02-08,85,Germany,1.1.0,2,2
9,3890477,2016-03-05,16:30:00.000,3,3,available,unscheduled,2023-07-24T09:18:59.971988,None,25,...,Germany,3980,Roger Schmidt,None,1967-03-13,85,Germany,1.1.0,2,2


Tutaj ponownie nie ma zbyt dużo przydatnych dla nas informacji (chyba, że będziemy chcieli też iść w predykcje biorąc pod uwage menadżera (?), bo jednak to też ma jakieś znaczenie - styl ofensywny vs defensywny).

Tu ważne dla nas na 100% jest match_id, bo możemy sobie wybrać eventy podczas tego meczu - czyli no wszystkie zdarzenia, co w kontekście naszego projektu jest jak foliowa czapeczka dla wujka, bez niej nie ruszy się na działke, tak jak my nie mamy po co zaczynać tematu.

In [66]:
len(pd.unique(matches_presentation_df['match_id']))

34

Przykładowo w tym sezonie Bundesligi mamy aż 34 mecze, co sprawia, że danych będziemy mieli w chuj i jeszcze trochę XD

## Creme de la creme - czyli to co jest dla nas ważne

In [11]:
concrete_match_df = sb.events(match_id=3890563) # wybór konkretnego meczu

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [12]:
concrete_match_df.head(20)

,50_50,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,...,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 442, 'lineup': [{'player': {'id'...",Bayer Leverkusen,904,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 433, 'lineup': [{'player': {'id'...",Ingolstadt,189,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Ingolstadt,189,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:00.000,Half Start,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Ingolstadt,189,00:00:00.000,Half Start,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:00.700,Pass,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:02.489,Pass,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:04.532,Pass,True
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:06.781,Pass,NaN


In [13]:
concrete_match_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2975 entries, 0 to 2974
Data columns (total 94 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   50_50                           4 non-null      object 
 1   ball_receipt_outcome            139 non-null    object 
 2   ball_recovery_recovery_failure  8 non-null      object 
 3   block_deflection                2 non-null      object 
 4   block_offensive                 2 non-null      object 
 5   block_save_block                2 non-null      object 
 6   carry_end_location              537 non-null    object 
 7   clearance_aerial_won            11 non-null     object 
 8   clearance_body_part             46 non-null     object 
 9   clearance_head                  25 non-null     object 
 10  clearance_left_foot             8 non-null      object 
 11  clearance_right_foot            13 non-null     object 
 12  counterpress                    15

Dla każdego meczu mamy dużo dużo danych różnego typu, tu  przy edzie  będzie trzeba przekminić, które z tych kolumn będą dla nas ważne, a które to "śmieci", bo tutaj mamy każdą akcję z meczu.

In [16]:
shots_df = concrete_match_df[concrete_match_df['type'] == 'Shot']

In [20]:
shots_df[['player', 'team', 'shot_statsbomb_xg', 'shot_outcome']]

,player,team,shot_statsbomb_xg,shot_outcome
2880,Moritz Hartmann,Ingolstadt,0.008072,Off T
2881,Pascal Groß,Ingolstadt,0.046675,Blocked
2882,Mathew Leckie,Ingolstadt,0.097066,Off T
2883,Mathew Leckie,Ingolstadt,0.977242,Goal
2884,Moritz Hartmann,Ingolstadt,0.267620,Off T
2885,Benjamin Hübner,Ingolstadt,0.029614,Off T
2886,Danny Vieira da Costa,Ingolstadt,0.018916,Wayward
2887,Charles Mariano Aránguiz Sandoval,Bayer Leverkusen,0.365020,Goal
2888,Kevin Kampl,Bayer Leverkusen,0.439847,Goal
2889,Moritz Hartmann,Ingolstadt,0.006357,Saved


In [25]:
print(len(shots_df))
print(len(shots_df[shots_df['team'] == 'Bayer Leverkusen']))
print(len(shots_df[shots_df['team'] == 'Ingolstadt']))

31
10
21


I tu nagle (co raczej spodziewane) okazuje się, że nie ma milionów strzałów, tylko, że łacznie było ich 31, z czego 10 dla Leverkusen, a 21 dla Ingolstadt.

## Ciekawostkowe rzeczy

Z ciekawych funkcji (choć raczej też nie jakoś bardzo przdyatnych dla nas) można sobie też wyświetlić lineup w danym meczu

In [54]:
def print_lineups(match_id, team1, team2):
  print(f"Lineup {team1}", sb.lineups(match_id=match_id)[team1])
  print(f"Lineup {team2}", sb.lineups(match_id=match_id)[team1])

In [55]:
print_lineups(match_id=3890563, team1='Bayer Leverkusen', team2='Ingolstadt')

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Lineup Bayer Leverkusen     player_id                        player_name   player_nickname  \
0        5555                      Julian Brandt              None   
1        7039                   Hakan Çalhanoğlu              None   
2        8210  Charles Mariano Aránguiz Sandoval  Charles Aránguiz   
3        8211                  Benjamin Henrichs              None   
4        8213                    Karim Bellarabi              None   
5        8219                      Admir Mehmedi              None   
6        8222                        Kevin Kampl              None   
7        8573                        Ömer Toprak              None   
8        8918                    Roberto Hilbert              None   
9        9363                André Ramalho Silva     André Ramalho   
10       9892                       Lukas Boeder              None   
11      11394                    Stefan Kießling              None   
12      15581                         Bernd Leno              None

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


## *Wybór meczu przypadkowy, Bundesliga mi jeszcze nie zapłaciła.